In [1]:
import os 
os.chdir("C:/Users/ASRENOVIN/Desktop/spatial-foundation-graph-transformer")

import yaml
import anndata as ad
import numpy as np

with open("configs/default.yaml") as f:
    cfg = yaml.safe_load(f)

adata = ad.read_h5ad("data/processed/adata_normalized.h5ad")
print(adata)

AnnData object with n_obs × n_vars = 3661 × 20955
    obs: 'in_tissue', 'array_row', 'array_col', 'n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg', 'log1p', 'pca', 'spatial'
    obsm: 'X_pca', 'spatial'
    varm: 'PCs'


In [2]:
# extract pca embedding

X_pca = adata.obsm["X_pca"].copy()

print("PCA embedding shape:", X_pca.shape)
print("Mean : ", X_pca.mean().round(4))
print("Std : ", X_pca.std().round(4))
print("Min : ", X_pca.min().round(4))
print("Max : ", X_pca.max().round(4))

PCA embedding shape: (3661, 50)
Mean :  0.0
Std :  1.203
Min :  -9.1818
Max :  14.5266


In [3]:
# save_pca_embedding
np.save("data/embeddings/X_pca.npy", X_pca)

print("Saved: data/embeddings/X_pca.npy")
print("Shape:", np.load("data/embeddings/X_pca.npy").shape)

Saved: data/embeddings/X_pca.npy
Shape: (3661, 50)


In [4]:
# write_embedding_module

base_code = '''from __future__ import annotations
from abc import ABC, abstractmethod
from typing import Any
import anndata as ad


class EmbeddingProvider(ABC):
    """
    Abstract base class for all embedding providers.
    Subclasses implement embed() to populate adata.obsm["X_embedding"].
    """

    def __init__(self, cfg: dict[str, Any]) -> None:
        self.cfg = cfg

    @abstractmethod
    def embed(self, adata: ad.AnnData) -> ad.AnnData:
        """
        Compute embeddings and store in adata.obsm["X_embedding"].

        Parameters
        ----------
        adata : AnnData  Normalized AnnData object.

        Returns
        -------
        AnnData  Same object with adata.obsm["X_embedding"] populated.
        """
        ...
'''

pca_code = '''from __future__ import annotations
from typing import Any
import numpy as np
import anndata as ad
from embeddings.base import EmbeddingProvider


class PCAEmbeddingProvider(EmbeddingProvider):
    """Uses the precomputed X_pca as node embeddings."""

    def embed(self, adata: ad.AnnData) -> ad.AnnData:
        if "X_pca" not in adata.obsm:
            raise ValueError("X_pca not found in adata.obsm. Run normalization first.")
        adata.obsm["X_embedding"] = adata.obsm["X_pca"].copy()
        np.save("data/embeddings/X_pca.npy", adata.obsm["X_embedding"])
        print(f"PCA embedding shape: {adata.obsm['X_embedding'].shape}")
        return adata
'''

factory_code = '''from __future__ import annotations
from typing import Any
import anndata as ad
from embeddings.base import EmbeddingProvider
from embeddings.pca_provider import PCAEmbeddingProvider


def get_embedding_provider(cfg: dict[str, Any]) -> EmbeddingProvider:
    """
    Return the correct EmbeddingProvider based on cfg["embeddings"]["provider"].
    Add new providers here as they are implemented.
    """
    provider = cfg["embeddings"]["provider"]
    if provider == "pca":
        return PCAEmbeddingProvider(cfg)
    raise NotImplementedError(f"Embedding provider '{provider}' is not implemented yet.")
'''

with open("src/embeddings/base.py", "w") as f:
    f.write(base_code)

with open("src/embeddings/pca_provider.py", "w") as f:
    f.write(pca_code)

with open("src/embeddings/factory.py", "w") as f:
    f.write(factory_code)

print("Created: src/embeddings/base.py")
print("Created: src/embeddings/pca_provider.py")
print("Created: src/embeddings/factory.py")

Created: src/embeddings/base.py
Created: src/embeddings/pca_provider.py
Created: src/embeddings/factory.py


In [5]:
# test the factory & persist embedding into adata
from src.embeddings.factory import get_embedding_provider

provider = get_embedding_provider(cfg)
adata = provider.embed(adata)

print("Embedding key in obsm:", list(adata.obsm.keys()))
print("X_embedding shape: ", adata.obsm["X_embedding"].shape)

adata.write_h5ad("data/processed/adata_embedded.h5ad")
print("Saved: data/processed/adata_embedded.h5ad")

PCA embedding shape: (3661, 50)
Embedding key in obsm: ['X_pca', 'spatial', 'X_embedding']
X_embedding shape:  (3661, 50)
Saved: data/processed/adata_embedded.h5ad
